# Chapter 1 — What Is an Agent? The Governed Loop

*Controlled decision systems vs language models.*

## Objective

A language model predicts tokens. An agent does something more dangerous and more useful — it acts.

This chapter draws the line between the two, then fixes the loop that separates them in reusable form. By the end you will have:
- Written a 20-line rule-based agent loop using only the standard library.
- Added a governance shim that rejects an unsafe action.
- Seen the difference between the naive loop (`observe → think → act`) and the governed loop (`observe → validate → plan → authorize → act → verify → log`).
- Rebuilt that governed loop on the `agentlab` primitives — `BaseAgent`, `Environment`, `run_loop` and `StepRecord` — the spine every later chapter extends.

## The naive loop

The simplest agent loop is `observe → think → act → observe`. It works for toy problems and fails the moment actions can be dangerous.

In [ ]:
from dataclasses import dataclass

@dataclass
class GridState:
    x: int
    y: int
    goal: tuple[int, int]
    forbidden: tuple[tuple[int, int], ...] = ()

def rule_based_step(s: GridState) -> str:
    '''Greedy step toward goal. No safety awareness.'''
    if s.x < s.goal[0]: return 'right'
    if s.x > s.goal[0]: return 'left'
    if s.y < s.goal[1]: return 'down'
    if s.y > s.goal[1]: return 'up'
    return 'stop'

def apply_move(s: GridState, action: str) -> GridState:
    dx, dy = {'right': (1, 0), 'left': (-1, 0), 'down': (0, 1), 'up': (0, -1)}.get(action, (0, 0))
    return GridState(s.x + dx, s.y + dy, s.goal, s.forbidden)

Now we put a forbidden cell on the agent's path. The naive loop walks through it.

In [ ]:
state = GridState(0, 0, (3, 3), forbidden=((2, 0), (3, 1)))
trace = [(state.x, state.y)]
for _ in range(20):
    a = rule_based_step(state)
    if a == 'stop':
        break
    state = apply_move(state, a)
    trace.append((state.x, state.y))

hit = [p for p in trace if p in state.forbidden]
print('trace:               ', trace)
print('forbidden cells hit: ', hit)

The agent walked right through the forbidden cells. This is the failure mode the rest of the book is designed to prevent.

## The governed loop

The governed loop wraps the same agent in an `authorize` step. Authorization is separate from the agent's logic — it gates actions.

In [ ]:
def authorize(s: GridState, action: str) -> str:
    proposed = apply_move(s, action)
    if (proposed.x, proposed.y) in s.forbidden:
        return 'deny'
    return 'allow'

def alternate(action: str, s: GridState) -> str:
    if action in ('right', 'left'):
        return 'down' if s.y < s.goal[1] else 'up'
    return 'right' if s.x < s.goal[0] else 'left'

state = GridState(0, 0, (3, 3), forbidden=((2, 0), (3, 1)))
trace = [(state.x, state.y)]
denied = []
for _ in range(40):
    a = rule_based_step(state)
    if a == 'stop':
        break
    if authorize(state, a) == 'deny':
        denied.append(((state.x, state.y), a))
        a = alternate(a, state)
        if authorize(state, a) == 'deny':
            print('no safe move; stopping'); break
    state = apply_move(state, a)
    trace.append((state.x, state.y))

print('governed trace:', trace)
print('denied:        ', denied)

## Taxonomy

| System | Main capability |
| --- | --- |
| Language model | predicts next token |
| Chatbot | responds to user |
| Tool-using model | calls external functions |
| Agent | pursues goals through actions |
| Governed agent | acts under explicit constraints, verification and audit |

This book is about turning the right column into a system. The rest of the chapters build each component of the governed loop carefully.

## Anti-patterns flagged here

- Treating the model's reasoning text as evidence (forward reference to Chapter 3).
- Assuming any model is a drop-in for any other. Model capability is a system parameter and dominates downstream behavior.

## The minimal agent loop

The hand-rolled loop above showed the shape. The rest of the chapter fixes it in reusable form, moving to the `agentlab` primitives: `BaseAgent`, `Environment`, `run_loop` and `StepRecord`. Every step is inspectable, and no intermediate state is hidden.

In [ ]:
from agentlab.core import (
    AgentState, BaseAgent, Finish, StepRecord, TaskSpec, ToolCall, run_loop,
)

## The agent

An agent subclasses `BaseAgent` and implements `propose_action(state) -> Action`. Default `update(state, action, observation)` appends the observation and advances the step counter.

In [ ]:
class GreedyAgent(BaseAgent):
    '''Proposes 'ping' tool calls until it has done `target` of them, then Finishes.'''
    def __init__(self, target: int) -> None:
        self.target = target
    def propose_action(self, state):
        if state.step >= self.target:
            return Finish(output={'pings': state.step})
        return ToolCall(tool_name='ping', arguments={})

## The environment

An environment is anything with a `step(action) -> dict` method. Production agents run actions through a `GovernedToolExecutor` (Chapter 6); here we use a tiny mock.

In [ ]:
class PingEnv:
    def step(self, action):
        return {'pong': True}

## Run the loop

`run_loop` is a generator. Each yielded `StepRecord` carries the state before, the action, the observation and the state after.

In [ ]:
task = TaskSpec(goal='do three pings')
state = AgentState(task=task)
records = list(run_loop(GreedyAgent(target=3), PingEnv(), state, max_steps=10))
print(f'{len(records)} step records yielded')
print(f'final status: {records[-1].state_after.status}')
print(f'final output: {records[-1].state_after.final_output}')

Each record is a self-contained snapshot of a step:

In [ ]:
for r in records:
    print(f'step {r.step}: action={r.action.kind:<10} obs={r.observation}')

## What the loop does and does not do

- It terminates on `Finish`, on `Escalate`, on `state.status != 'running'`, or on `max_steps`.
- It does **not** call any LLM. The agent is a plain Python object.
- Token, time and tool-call budgets are added in Chapter 7. Governance gates and audit are added in Chapter 12. Both extend this loop without rewriting it.

## Anti-patterns flagged here

- Loops that hide intermediate state.
- Mixing the loop driver with the policy.
- Treating the LLM call as the loop.

In [ ]:
# Self-check --- both halves of the chapter
# governed grid loop: reaches the goal, avoids the forbidden cells, records a denial
assert (2, 0) not in trace and (3, 1) not in trace, 'governed loop entered a forbidden cell'
assert trace[-1] == (3, 3), 'governed loop should reach the goal'
assert denied, 'governed loop should have recorded at least one denial'
# minimal agentlab loop: typed step records, terminates done, correct output
assert all(isinstance(r, StepRecord) for r in records)
assert records[-1].state_after.status == 'done'
assert records[-1].state_after.final_output == {'pings': 3}
print('OK')